# 08 — Visualising training so you can debug it

You can't fix what you can't see. Five plots every training loop should produce.

## Roadmap
1. Loss curves (train vs val) — overfit detector
2. **Learning-rate vs loss** — the LR-finder trick
3. **Gradient flow** — is every layer learning?
4. **Confusion matrix heatmap** — which classes the model confuses
5. **Activation maps** — what the model "looks at"


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Loss curves — the #1 diagnostic


In [ ]:
import numpy as np, matplotlib.pyplot as plt
np.random.seed(0)
epochs = np.arange(1, 31)
train_loss = 1.2 * np.exp(-0.15 * epochs) + 0.05 * np.random.randn(30) + 0.1
val_loss   = train_loss + 0.1 + 0.005 * (epochs ** 1.5)   # diverges late
plt.figure(figsize=(8, 4))
plt.plot(epochs, train_loss, label="train")
plt.plot(epochs, val_loss, label="val")
plt.axvline(np.argmin(val_loss), linestyle="--", color="red", alpha=0.5)
plt.text(np.argmin(val_loss)+0.5, 0.5, "early stop here", color="red")
plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss"); plt.title("Classic overfit"); plt.show()


**What you're looking for:**
- Train ↓ steadily but val flat → underfit. Try bigger model, longer training, less regularisation.
- Both ↓ then val ↑ → overfit. Early-stop at val minimum, more augmentation, more dropout.
- Both ↓ together → still fitting. Keep training.

## 2. Learning-rate finder

Sweep LR from 1e-7 to 10 over a few hundred batches, plot loss vs LR. Pick the LR where loss drops fastest (typically ~1/10 of where it diverges).


In [ ]:
# Synthesise a typical "LR finder" curve to demo
lrs = np.logspace(-7, 1, 200)
loss = 1 / (1 + lrs * 100) + 0.05 * np.random.randn(200)
loss[lrs > 1] = loss[lrs > 1] * (1 + lrs[lrs > 1])    # diverges at high LR
plt.figure(figsize=(8, 4))
plt.semilogx(lrs, loss); plt.axvspan(1e-3, 1e-1, alpha=0.2, color="green")
plt.xlabel("learning rate"); plt.ylabel("loss")
plt.title("Pick the LR where loss falls fastest (green band)"); plt.grid(True); plt.show()


## 3. Gradient-flow plot

For each layer, plot `||grad||`. If early layers are flat at 0, you have vanishing-gradient. If late layers spike to huge values, exploding-gradient.


In [ ]:
# Synthetic example — a healthy vs unhealthy gradient flow
layers = list(range(1, 13))
healthy = np.exp(-0.05 * np.arange(12)) * (1 + 0.1*np.random.randn(12))
unhealthy = np.exp(-0.4 * np.arange(12))
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].bar(layers, healthy); ax[0].set_title("healthy: gradients ~uniform"); ax[0].set_ylim(0, 1.2)
ax[1].bar(layers, unhealthy); ax[1].set_title("vanishing: early layers near zero"); ax[1].set_ylim(0, 1.2)
for a in ax: a.set_xlabel("layer index"); a.set_ylabel("‖grad‖")
plt.show()


## 4. Confusion-matrix heatmap

For multi-class classification, this shows you which pairs of classes the model confuses. A diagonal-heavy matrix = good model. Off-diagonal hot spots = systematic confusion you can target with more data or better features.


In [ ]:
# Synthetic 10-class confusion matrix
np.random.seed(0)
cm = np.zeros((10, 10), dtype=int)
for true in range(10):
    cm[true, true] = np.random.randint(40, 60)              # most correct
    for pred in range(10):
        if pred != true:
            cm[true, pred] = np.random.randint(0, 10)
plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.colorbar(label="count")
plt.xlabel("predicted"); plt.ylabel("true"); plt.title("Confusion matrix")
for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i, j], ha="center", va="center",
                  color="white" if cm[i, j] > 30 else "black", fontsize=8)
plt.show()


## 5. Activation maps — what the CNN looks at

For a trained CNN, take an intermediate feature map and visualise it. Tells you whether the conv filters have actually learned anything semantic (edges, parts) or are stuck on noise.

(Real production tooling uses Grad-CAM and friends to highlight pixels that most contributed to a particular prediction. The principle: backprop the score of one class back to the input and visualise the gradient magnitude.)


In [ ]:
# Toy demo: 4 conv kernels applied to a synthetic image
img = np.zeros((32, 32), dtype=np.float32)
img[8:16, 8:24] = 1.0   # horizontal bar
img[8:24, 14:18] = 1.0  # vertical bar — forms a "T"

kernels = {
    "vertical edge":   np.array([[-1, 0, 1]] * 3),
    "horizontal edge": np.array([[-1, -1, -1], [0, 0, 0], [1, 1, 1]]),
    "corner":          np.array([[ 0,  1, 0],
                                  [ 1, -4, 1],
                                  [ 0,  1, 0]]),
    "blur":            np.ones((3, 3)) / 9,
}

def conv2d_quick(im, k):
    from scipy.signal import correlate2d  # NB: only used here for speed
    return correlate2d(im, k, mode="same")

try:
    from scipy.signal import correlate2d
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    axes[0].imshow(img, cmap="gray"); axes[0].set_title("input"); axes[0].axis("off")
    for ax, (name, k) in zip(axes[1:], kernels.items()):
        ax.imshow(conv2d_quick(img, k.astype(np.float32)), cmap="RdBu"); ax.set_title(name); ax.axis("off")
    plt.show()
except ImportError:
    print("scipy not available; skipping plot.")


**Next:** the same models, now in PyTorch — line-by-line contrast.
